# 01 · Data pipeline
SEC 10-K filings → clean text → `RecursiveCharacterTextSplitter(500, 50)` → MiniLM embeddings (d = 384, L2-normalised) → FAISS index.  Re-run with `python -m src.data_pipeline` and `python -m src.embeddings`.

In [1]:
import sys, json, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 120)

In [2]:
from src.embeddings import load_chunks, load_queries
chunks = pd.DataFrame(load_chunks()); queries = pd.DataFrame(load_queries())
print(len(chunks), 'chunks'); chunks.groupby(['ticker','company','year']).size().rename('chunks').to_frame()

6368 chunks


,,,chunks
ticker,company,year,
AAPL,Apple Inc.,2025,555
AMZN,Amazon.com Inc.,2026,754
GOOGL,Alphabet Inc.,2025,920
META,Meta Platforms Inc.,2026,1305
MSFT,Microsoft Corp.,2026,853
NVDA,NVIDIA Corp.,2026,916
TSLA,Tesla Inc.,2026,1065


In [3]:
chunks.char_count.describe()

count    6368.000000
mean      359.630967
std       107.278735
min       100.000000
25%       284.000000
50%       381.000000
75%       454.000000
max       499.000000
Name: char_count, dtype: float64

In [4]:
for t in chunks.sample(3, random_state=1).text: print(t, '\n---')

In addition, regulators and lawmakers are increasingly focused on controlling additional aspects of the operations of technology companies and companies they have characterized to be online “gatekeepers” through the application of existing regulations and laws and the adoption of new regulations and laws, which increases our compliance costs and limits the operation of our business. 
---
Governmental Programs, Incentives and Regulations
Globally, the ownership of our products by our customers is impacted by various government credits, incentives, and policies. Our business and products are also subject to numerous governmental regulations that vary among jurisdictions. 
---
• The EU's Digital Markets Act, which requires in-scope companies to obtain user consent for combining data across certain products and require search engines to share anonymized data with rival companies, among other changes; and the EU Data Act, which introduces new data portability requirements with respect to co

## Evaluation query sets

In [5]:
display(queries.label.value_counts().to_frame())
queries.groupby('label').head(3)[['query_id','label','query_text']]

,count
label,
IN_DISTRIBUTION,150
NEAR_OOD,100
FAR_OOD,100


,query_id,label,query_text
0,ID_0001,IN_DISTRIBUTION,What are Apple's principal products and services disclosed in Item 1 of Form 10-K?
1,ID_0002,IN_DISTRIBUTION,How does Apple define its Services segment net sales in Management's Discussion and Analysis?
2,ID_0003,IN_DISTRIBUTION,What are the primary factors affecting Apple's gross margin according to its financial statements?
150,NEAR_OOD_0001,NEAR_OOD,What was Apple's total net revenue from the Apple Car Titan project in fiscal year 2024?
151,NEAR_OOD_0002,NEAR_OOD,How much profit did Apple report from its holographic eyewear subscription service?
152,NEAR_OOD_0003,NEAR_OOD,What were Apple's capital expenditures for nuclear fusion powered data centers in 2023?
250,FAR_OOD_0001,FAR_OOD,The character Wario was first introduced in which game of the Mario video game franchise?
251,FAR_OOD_0002,FAR_OOD,Which of these African regions does *not* directly border the Democratic Republic of The Congo?
252,FAR_OOD_0003,FAR_OOD,Who designed the Chupa Chups logo?


## Synthetic in-distribution questions (doc2query)
Used to fit / calibrate the *query-fit* guardrail; see `src/synthetic_queries.py`.

In [6]:
syn = pd.DataFrame(json.load(open('data/synthetic_queries.json', encoding='utf-8')))
print(len(syn), 'synthetic questions'); syn.sample(8, random_state=0)

12668 synthetic questions


,chunk_index,chunk_id,query_text
2400,1204,MSFT_chunk_00649,how many level do you have to be to use to find fair value
8438,4240,META_chunk_00242,what are the changes
643,322,AAPL_chunk_00322,what is the current principal value of bank note
6289,3158,NVDA_chunk_00830,who was the plaintiffs in nvidia arbitration
1543,773,MSFT_chunk_00218,can open source systems contain vulnerabilities
10866,5458,TSLA_chunk_00155,what kind of problems does it occur for suppliers to deliver material
7350,3692,AMZN_chunk_00448,how much do we have on consolidated balance sheets
2604,1306,MSFT_chunk_00751,when do you purchase stock


## Embedding geometry
Passages and questions occupy different regions of the embedding space (query–passage asymmetry) — the main reason the guardrail is fit on questions.

In [7]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from src.plotting import SPLIT_COLORS, SPLIT_NAMES
X = np.load('data/corpus_embeddings.npy'); Q = np.load('data/query_embeddings.npy'); S = np.load('data/synthetic_query_embeddings.npy')
pca = PCA(2, random_state=0).fit(np.vstack([X, S]))
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(*pca.transform(X).T, s=3, c='#c3c2b7', label='corpus chunks')
ax.scatter(*pca.transform(S[::4]).T, s=3, c='#6250d6', alpha=.4, label='doc2query questions')
labels = queries.label.values
for l in SPLIT_COLORS:
    ax.scatter(*pca.transform(Q[labels==l]).T, s=16, c=SPLIT_COLORS[l], edgecolor='white', lw=.4, label=SPLIT_NAMES[l])
ax.legend(fontsize=8); ax.set_title('PCA of MiniLM embeddings'); plt.show()

C:\Users\Sai Krishna\AppData\Local\Temp\ipykernel_26128\2718895869.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(fontsize=8); ax.set_title('PCA of MiniLM embeddings'); plt.show()
